In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cwd = os.getcwd()
if cwd.endswith('notebook'):
    os.chdir('..')
    cwd = os.getcwd()

In [2]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('./data')
assert data_folder.is_dir()

figures_folder = Path('./figures')
assert figures_folder.is_dir()

In [27]:
gtdb_metadata = pd.read_csv(data_folder / 'gtdb_metadata.csv', index_col='ncbi_accession')
archaeal_accessions = set(gtdb_metadata[gtdb_metadata['domain'] == 'Archaea'].index)

In [28]:
archaeal_hits = pd.read_csv(data_folder / 'pg_synthesis' / 'pg_synthesis_archaeal_hits.csv')
print(f'Number of archaeal hits: {len(archaeal_hits):,}')
archaeal_hits.head()

Number of archaeal hits: 42,070


,target_name,accession,query_name,accession_query,full_evalue,full_score,full_bias,dom_evalue,dom_score,dom_bias,exp,reg,clu,ov,env,dom,rep,inc
0,MCI5866507.1@GCA_022768985.1,GCA_022768985.1,newDdlB,-,9.100000e-92,320.2,0.3,1.000000e-91,320.0,0.3,1.0,1,0,0,1,1,1,1
1,WTHQ01000015.1_4@GCA_011523055.1,GCA_011523055.1,newDdlB,-,1.100000e-81,287.1,0.1,1.200000e-81,286.9,0.1,1.0,1,0,0,1,1,1,1
2,PIN78533.1@GCA_002762865.1,GCA_002762865.1,newDdlB,-,5.200000e-74,261.8,0.0,6.500000e-74,261.5,0.0,1.0,1,0,0,1,1,1,1
3,JALRLN010000114.1_3@GCA_023254445.1,GCA_023254445.1,newDdlB,-,2.300000e-67,240.0,0.0,2.800000e-67,239.7,0.0,1.0,1,0,0,1,1,1,1
4,NQU98779.1@GCA_013202845.1,GCA_013202845.1,newDdlB,-,3.900000e-65,232.7,0.7,4.800000e-65,232.4,0.7,1.0,1,0,0,1,1,1,1


In [36]:
top_hits = archaeal_hits[
    archaeal_hits['full_evalue'] <= 1e-6
][
    ['accession', 'target_name', 'query_name', 'full_score']
].sort_values(
    ['accession', 'query_name', 'full_score'],
    ascending=[True, True, False],
).drop_duplicates(
    ['accession', 'query_name']
)
print(f'Number of hits: {len(top_hits):,}')
top_hits.head()

Number of hits: 15,429


,accession,target_name,query_name,full_score
16619,GCA_000008085.1,AAR38988.1@GCA_000008085.1,newFtsZ,380.2
844,GCA_000016605.1,ABP96130.1@GCA_000016605.1,newDdlB,86.9
25262,GCA_000016605.1,ABP96273.1@GCA_000016605.1,newMraY,100.1
28650,GCA_000016605.1,ABP96011.1@GCA_000016605.1,newMurA,45.3
28808,GCA_000016605.1,ABP94902.1@GCA_000016605.1,newMurB,45.2


In [37]:
grouped_df = top_hits[['query_name', 'accession']].groupby('query_name').nunique().sort_values('accession', ascending=False)
grouped_df['percent'] = (100 * grouped_df['accession'] / len(archaeal_accessions)).round(1)
grouped_df

,accession,percent
query_name,,
newFtsZ,3260,88.0
newDdlB,2670,72.0
newMraY,1902,51.3
newMurA,1792,48.4
newMurE,1341,36.2
newMurF,1328,35.8
newMurG,1141,30.8
newMurC,915,24.7
newMurD,619,16.7


In [43]:
top_hits_with_phylum = pd.merge(
    top_hits,
    gtdb_metadata[
        gtdb_metadata['domain'] == 'Archaea'
    ].reset_index()[
        ['ncbi_accession', 'gtdb_phylum']
    ].rename(columns={'ncbi_accession': 'accession'}),
    on='accession',
    how='left',
)
top_hits_with_phylum.head()

,accession,target_name,query_name,full_score,gtdb_phylum
0,GCA_000008085.1,AAR38988.1@GCA_000008085.1,newFtsZ,380.2,Nanoarchaeota
1,GCA_000016605.1,ABP96130.1@GCA_000016605.1,newDdlB,86.9,Thermoproteota
2,GCA_000016605.1,ABP96273.1@GCA_000016605.1,newMraY,100.1,Thermoproteota
3,GCA_000016605.1,ABP96011.1@GCA_000016605.1,newMurA,45.3,Thermoproteota
4,GCA_000016605.1,ABP94902.1@GCA_000016605.1,newMurB,45.2,Thermoproteota


In [45]:
top_hits_with_phylum[
    ['gtdb_phylum', 'query_name', 'accession']
].groupby(['gtdb_phylum', 'accession']).nunique().reset_index()[
    ['gtdb_phylum', 'query_name']
].groupby('gtdb_phylum').mean().sort_values('query_name', ascending=False)

,query_name
gtdb_phylum,
Methanobacteriota,8.953333
Altiarchaeota,5.961538
Iainarchaeota,5.588235
Halobacteriota,5.182679
JACRDV01,5.000000
Thermoplasmatota,4.396985
Methanobacteriota_B,4.368421
Hydrothermarchaeota,4.312500
Asgardarchaeota,3.888298
